In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import contextily as ctx
import numpy as np
import plotly.graph_objects as go
import pandas as pd
import plotly.express as px
import pyproj
from plotly.subplots import make_subplots
pd.options.mode.chained_assignment = None
path_to_save = 'C:/Users/sylva/OneDrive/Bureau/senat/plot/'
#FR: © EuroGeographics pour les limites administratives
#https://ec.europa.eu/eurostat/web/gisco/geodata/administrative-units/communes

In [ ]:
senateurs = pd.read_csv("C:/Users/sylva/OneDrive/Bureau/senat/Data/senateur_for_map.csv")


In [ ]:
years = list(range(1789, 1816))

In [ ]:
# Reste dans la ville sauf mention contraire 
# prendre en compte les morts
#prendre lieu précédent par défaut
# agréger les différentes participations assemblées révolution
# grand conseil d'administration : vérifier si récupération de plusieurs dates 
# nombre de décorations 

In [ ]:
# activité famille 
# interprétation des cartes

In [ ]:
from geopy.geocoders import Nominatim
geolocator = Nominatim(user_agent="test_senateur")

In [ ]:
from geopy.extra.rate_limiter import RateLimiter
from geopy.distance import geodesic
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

In [ ]:
dic_lieux = {}

In [ ]:
def geocode_to_apply(x):
    if x in dic_lieux.keys():
        return dic_lieux[x]
    else:
        result = geocode(x)
        dic_lieux[x]=result
        return result

In [ ]:
geocode_paris = geocode('Paris')
latitude_paris = geocode_paris.latitude
longitude_paris = geocode_paris.longitude
def compute_distance(x):
    if not x:
        return None
    return (geodesic((latitude_paris, longitude_paris), (x.latitude, x.longitude)).kilometers)

In [ ]:
def compute_distance_to_birth(personne, year):
    if (not personne['latitude'+str(annee)]) or pd.isna(personne['latitude'+str(annee)]) or pd.isna(personne['latitude_naiss']):
        return None
    return (geodesic((personne['latitude_naiss'], personne['longitude_naiss']), 
                     (personne['latitude'+str(annee)], personne['longitude'+str(annee)])).kilometers)

def get_country(personne, year):
    if not personne:
        return None
    return personne.address.split(', ')[-1]

In [ ]:
dic_annee={}
pd.options.mode.chained_assignment = None
senateurs["result_naiss"] = senateurs["ville naissance"].apply(geocode_to_apply)
senateurs['latitude_naiss'] = senateurs['result_naiss'].apply(lambda x : x.latitude if x else None)
senateurs['longitude_naiss'] = senateurs['result_naiss'].apply(lambda x : x.longitude if x else None)
senateurs["result_AR_1"] = senateurs["lieu AR 1"].apply(geocode_to_apply)
senateurs['latitude_AR_1'] = senateurs['result_AR_1'].apply(lambda x : x.latitude if x else None)
senateurs['longitude_AR_1'] = senateurs['result_AR_1'].apply(lambda x : x.longitude if x else None)
senateurs["result_AR_2"] = senateurs["lieu AR 2"].apply(geocode_to_apply)
senateurs['latitude_AR_2'] = senateurs['result_AR_2'].apply(lambda x : x.latitude if x else None)
senateurs['longitude_AR_2'] = senateurs['result_AR_2'].apply(lambda x : x.longitude if x else None)
senateurs["result_senatoreries"] = senateurs["senatoreries"].apply(geocode_to_apply)
senateurs['latitude_senatoreries'] = senateurs['result_senatoreries'].apply(lambda x : x.latitude if x else None)
senateurs['longitude_senatoreries'] = senateurs['result_senatoreries'].apply(lambda x : x.longitude if x else None)
for annee in years:#senateurs[senateurs[str(annee)].notna() & 
    senateurs["nom_local"+str(annee)].fillna('', inplace = True)
    senateurs['result'+str(annee)] = senateurs["nom_local"+str(annee)].apply(geocode_to_apply)
    senateurs['distance'+str(annee)] = senateurs['result'+str(annee)].apply(compute_distance)
    senateurs['latitude'+str(annee)] = senateurs['result'+str(annee)].apply(lambda x : x.latitude if x else None)
    senateurs['longitude'+str(annee)] = senateurs['result'+str(annee)].apply(lambda x : x.longitude if x else None)
    senateurs['distance_to_naiss'+str(annee)] = senateurs.apply(compute_distance_to_birth, axis = 1, args = [annee])
    

In [ ]:
for annee in years:
    senateurs['country_'+str(annee)] = senateurs['result'+str(annee)].apply(get_country, args = [annee])
    senateurs['international_'+str(annee)] = senateurs['country_'+str(annee)] != "France"
senateurs['country_naiss'] = senateurs['result_naiss'].apply(get_country, args = [annee])
senateurs['international_naiss'] = senateurs['country_naiss'] != "France"
senateurs['country_AR_1'] = senateurs['result_AR_1'].apply(get_country, args = [annee])
senateurs['international_AR_1'] = senateurs['country_AR_1'] != "France"
senateurs['country_AR_2'] = senateurs['result_AR_2'].apply(get_country, args = [annee])
senateurs['international_AR_2'] = senateurs['country_AR_2'] != "France"



In [ ]:
senateurs.to_csv("C:/Users/sylva/OneDrive/Bureau/senat/Data/senateur_with_lat.csv")

In [ ]:
senateurs = pd.read_csv("C:/Users/sylva/OneDrive/Bureau/senat/Data/senateur_with_lat.csv")

In [ ]:
senateurs['francais'] = senateurs['nationalite']=='français'

In [ ]:
dic_annee={}
for annee in years:#senateurs[senateurs[str(annee)].notna() &
    list_name = ["nom_local"+str(annee), "activite"+str(annee), "nom", "annee nomin", #"position_sociale_restreint",
                 'latitude'+str(annee), 'longitude'+str(annee), 'distance_to_naiss'+str(annee),
                'senatoreries', 'francais', 'jours conges 1800', 'jours conges 1801', 'jours conges 1802', #'senatorerie',
                 'jours conges 1803',  'nationalite', 'nation_short', 'position', 'noblesse AR', 'revolution trajectory']
    senateur_year = senateurs[list_name]
    senateur_year["currently in office"] = (senateur_year["annee nomin"]<=annee).astype(int)
    senateurs['currently in office'+str(annee)] = (senateurs["annee nomin"]<=annee).astype(int)
    dic_annee[annee]=senateur_year

In [ ]:

for annee in years:#senateurs[senateurs[str(annee)].notna() & 
    dic_annee[annee]["nom_local"] = dic_annee[annee]["nom_local" + str(annee)]
    dic_annee[annee]["nombre_senateurs"] = dic_annee[annee].groupby("nom_local").nom.transform("nunique")


In [ ]:
annee = 'AR 1'
list_name = ["lieu AR 1", "nom", "annee nomin", #"position_sociale_restreint",
             'latitude_AR_1', 'longitude_AR_1',
            'senatoreries', 'francais', 'jours conges 1800', 'jours conges 1801', 'jours conges 1802', 
             'jours conges 1803',  'nationalite', 'nation_short', 'position']
senateur_year = senateurs[list_name]
dic_annee[annee]=senateur_year
dic_annee[annee]["nom_local"] = dic_annee[annee]["lieu AR 1"]
dic_annee[annee]["nombre_senateurs"] = dic_annee[annee].groupby("nom_local").nom.transform("nunique")
annee = 'AR_2'
list_name = ["lieu AR 2", "nom", "annee nomin", #"position_sociale_restreint",
             'latitude_AR_2', 'longitude_AR_2',
            'senatoreries', 'francais', 'jours conges 1800', 'jours conges 1801', 'jours conges 1802', 
             'jours conges 1803',  'nationalite', 'nation_short', 'position']
senateur_year = senateurs[list_name]
dic_annee[annee]=senateur_year
dic_annee[annee]["nom_local"] = dic_annee[annee]["lieu AR 2"]
dic_annee[annee]["nombre_senateurs"] = dic_annee[annee].groupby("nom_local").nom.transform("nunique")
annee = 'naiss'
list_name = ["ville naissance", "nom", "annee nomin", #"position_sociale_restreint",
             'latitude_naiss', 'longitude_naiss', 
            'senatoreries', 'francais', 'jours conges 1800', 'jours conges 1801', 'jours conges 1802', 
             'jours conges 1803',  'nationalite', 'nation_short', 'position']
senateur_year = senateurs[list_name]
dic_annee[annee]=senateur_year
dic_annee[annee]["nom_local"] = dic_annee[annee]["ville naissance"]
dic_annee[annee]["nom_localnaiss"] = dic_annee[annee]["ville naissance"]
dic_annee[annee]["latitudenaiss"] = dic_annee[annee]["latitude_naiss"]
dic_annee[annee]["longitudenaiss"] = dic_annee[annee]["longitude_naiss"]
dic_annee[annee]["nombre_senateurs"] = dic_annee[annee].groupby("nom_local").nom.transform("nunique")
annee = 'senatoreries'
list_name = ["senatoreries", "nom", "annee nomin", #"position_sociale_restreint",
             'latitude_senatoreries', 'longitude_senatoreries',
             'francais', 'jours conges 1800', 'jours conges 1801', 'jours conges 1802', 
             'jours conges 1803',  'nationalite', 'nation_short', 'position']
senateur_year = senateurs[list_name]
dic_annee[annee]=senateur_year
dic_annee[annee]["nom_local"] = dic_annee[annee]["senatoreries"]
dic_annee[annee]["nom_localsenatoreries"] = dic_annee[annee]["senatoreries"]
dic_annee[annee]["latitudesenatoreries"] = dic_annee[annee]["latitude_senatoreries"]
dic_annee[annee]["longitudesenatoreries"] = dic_annee[annee]["longitude_senatoreries"]
dic_annee[annee]["nombre_senateurs"] = dic_annee[annee].groupby("nom_local").nom.transform("nunique")


In [ ]:
dic_annee[1800].columns

In [ ]:
year=1795

In [ ]:
minx, miny, maxx, maxy = -10.4477, 34.5949, 34.3694, 71.0313

In [ ]:
def update_layout_default(fig, year = 1800, title = 'Senateurs'):
        return fig.update_layout(
        height = 500,
        width = 500,
        title = go.layout.Title(
            text = title),
        geo = go.layout.Geo(
            resolution = 110,
            scope = 'europe',
            showframe = False,
            showcoastlines = True,
            landcolor = "rgb(229, 229, 229)",
            #countrycolor = None,
            showcountries = False,
            coastlinecolor = "white",
            projection_type = 'natural earth',#'mercator
            lonaxis_range= [ minx, maxx -5],
            lataxis_range= [ miny, maxy -4 ],
        ),
        geo2 = go.layout.Geo(
            #scope = 'europe',
            showframe = False,
            landcolor = "rgb(229, 229, 229)",
            showcountries = False,
            #domain = dict(x = [ 0, 0.6 ], y = [ 0, 0.6 ]),
            bgcolor = 'rgba(255, 255, 255, 0.0)',
        )#,
        #legend_traceorder = 'reversed'
    )
    

In [ ]:
colors = ['#1f77b4',  # muted blue
    '#ff7f0e',  # safety orange
    '#2ca02c',  # cooked asparagus green
    '#d62728',  # brick red
    '#9467bd',  # muted purple
    '#8c564b',  # chestnut brown
    '#e377c2',  # raspberry yogurt pink
    '#7f7f7f',  # middle gray
    '#bcbd22',  # curry yellow-green
    '#17becf', 'red', 'blue', 'green', 'brown', 'purple', 'deepskyblue', 'gold', 'coral', 'orange',
            'darkkhaki', 'darkmagenta', 'darkolivegreen', 'darkorange',
            'darkorchid', 'darkred', 'darksalmon', 'darkseagreen',
            'darkslateblue', 'darkslategray',
            'darkturquoise', 'darkviolet', 'deeppink', 'dimgrey', 'dodgerblue', 'firebrick',
            'floralwhite', 'forestgreen', 'fuchsia', 'gainsboro', 'gold', 'goldenrod', 'gray', 'grey']
markers = ['circle', 'square', 'diamond', 'cross', 'x', 'triangle-up', 'pentagon', 'hexagram',
           'star', 'hourglass', 'bowtie', 'asterisk', 'hash', 'triangle-down', 'octagon', 'cross-open', 'triangle-nw-dot',
          'pentagon-dot', 'hexagon-dot']

In [ ]:

def plot_from_year(year = 1800, column_to_separe='en exercice', column_to_separe2=None, save = False, title = None, include_paris = False):#, position = [1, 1], big_fig = make_subplots(
        #rows=1, cols=1)):
    #row= position[0]
    #col = position[1]
    df = dic_annee[year]
    if not include_paris:
        df= df[~df["nom_local"+str(year)].isin(["Paris", "paris"])]

    fig = go.Figure()
    i = 0
    if column_to_separe is None:
        df["nombre_senateurs"] = df.groupby("nom_local").nom.transform("nunique").fillna(0)
        fig.add_trace(
                go.Scattergeo(
                lon = df['longitude'+str(year)],
                lat = df['latitude'+str(year)],
                text = df['nom_local'],
                showlegend = True,
                opacity = 1,
                marker = dict(
                    size = df['nombre_senateurs'].apply(np.sqrt)*3,
                    line_width = 0,
                )))
    else:
        for value in df[column_to_separe].unique():
            if (value!= value):
                value = 'Inconnu'
            df_part = df[df[column_to_separe]==value]
            if column_to_separe2 is None:
                df_part["nombre_senateurs"] = df_part.groupby("nom_local").nom.transform("nunique").fillna(0)
                fig.add_trace(
                        go.Scattergeo(
                        lon = df_part['longitude'+str(year)],
                        lat = df_part['latitude'+str(year)],
                        text = df_part['nom_local'],
                        showlegend = True,
                        name = str(value),
                        opacity = 0.7,
                        marker = dict(
                            size = df_part['nombre_senateurs'].apply(np.sqrt)*3,
                            line_width = 0,
                            color = colors[i],
                        )))#, row=row, col=col)
            else:
                j=0
                for value2 in df[column_to_separe2].unique():
                    if (value2!= value2):
                        value2 = 'Inconnu'
                    df_part2 = df_part[df_part[column_to_separe2]==value2]
                    df_part2["nombre_senateurs"] = df_part2.groupby("nom_local").nom.transform("nunique")
                    df_part2["nombre_senateurs"]= df_part2["nombre_senateurs"].fillna(0)
                    fig.add_trace(go.Scattergeo(
                            lon = df_part2['longitude'+str(year)],
                            lat = df_part2['latitude'+str(year)],
                            text = df_part2['nom_local'],
                            opacity = 0.7,
                            showlegend = True,
                            name = str(value2),
                            marker = dict(
                                size = df_part2['nombre_senateurs'].apply(np.sqrt)*3,
                                line_width = 0,
                                symbol=markers[j],
                                color = colors[i],
                            ))#, row=row, col=col)
                                 )
                    j+=1
            i+=1
    if title:
        update_layout_default(fig, year = year, title = title)
    else:
        update_layout_default(fig, year = year, title = 'Senateurs en {}'.format(year))
    if save:
        fig.write_image(save)

    return fig

In [ ]:
plot_from_year(year = 'naiss', column_to_separe = 'nation_short', 
               title = 'Senators birthplace', include_paris = True,
              save = path_to_save + 'lieu naissance'+'.png')

In [ ]:
plot_from_year(year = 'senatoreries', column_to_separe = None, 
               title = 'Senatoreries', include_paris = True,
              save = path_to_save + 'senatoreries'+'.png')

In [ ]:
plot_from_year(year = 1800, column_to_separe = 'revolution trajectory', 
               title = 'Revolution trajectory 1800', include_paris = True,
              save = path_to_save + 'activity_revolution_1800'+'.png')#'revolution trajectory'

In [ ]:
for year in years:
    plot_from_year(year = year, column_to_separe = 'revolution trajectory', 
               title = 'Revolution trajectory '+str(year), include_paris = True,
              save = path_to_save + 'maps/activity revolution/' + 'activity_revolution'+str(year)+'.png')

In [ ]:
plot_from_year(year = 1800, column_to_separe = 'activite1800', 
               title = 'Activity 1800', include_paris = True,
              save = False)#path_to_save + 'activity1800'+'.png'

In [ ]:
for year in years:
    plot_from_year(year = year, save = path_to_save + str(year)+'.png')


In [ ]:
for year in years:
    plot_from_year(year = year, column_to_separe = 'noblesse AR', save = path_to_save + 'maps/noblesse/noblesse' + str(year)+'.png')

In [ ]:
path_to_save

In [ ]:
plot_from_year(year = 1800, column_to_separe = 'jours conges 1800', include_paris=True, save = path_to_save + '1800 conges.png')


In [ ]:
plot_from_year(year = 1813, column_to_separe = 'nation_short', title = 'Senateurs selon leur origine en 1813',
               save = path_to_save + '1813 nations.png')


In [ ]:
plot_from_year(year = 1808, column_to_separe = 'nation_short', title = 'Senateurs selon leur origine en 1808',
               save = path_to_save + '1808 nations.png')

In [ ]:
plot_from_year(year = 1789, save = 'https://onedrive.live.com/?authkey=%21AB5dud8S3DBRfXE&id=4C6BFAF1D3A2778E%21211&cid=4C6BFAF1D3A2778E/plot_1789.png')


In [ ]:
plot_from_year(year = 1812, column_to_separe = 'position', title = 'Senateurs selon leur origine en 1808',
               save = path_to_save + '1808 position.png')

In [ ]:
plot_from_year(year = 1806, column_to_separe = 'position_sociale')

In [ ]:
def create_indicators_on_column(colname, only_en_exercice = True):
    df_to_complete = pd.DataFrame(columns = ["year", colname, "mean distance", "mean distance to birthplace", "share not in Paris"])
    
    for annee in years:
        if only_en_exercice:
            senateurs_to_keep = senateurs[(senateurs["annee nomin"]<=annee) & (senateurs["annee deces"]>annee)]
        else:
            senateurs_to_keep = senateurs[senateurs["annee deces"]>annee]
        if colname + str(annee) in senateurs_to_keep.columns:
            col_year = colname + str(annee)
        else:
            col_year = colname
        for type_senator in senateurs_to_keep[col_year].unique():
            senateur_type = senateurs_to_keep[senateurs_to_keep[col_year]==type_senator]
            liste_dist = senateur_type["distance"+str(annee)]
            liste_dist_naiss = senateur_type['distance_to_naiss'+str(annee)]
            liste_hors_paris = ~senateur_type["nom_local"+str(annee)].isin(['Paris', 'paris'])
            if len(senateur_type)>0:
                value = sum(liste_hors_paris)/len(liste_hors_paris)
                dic_line = {"year" : annee, colname: type_senator, "share not in Paris": value,
                           "mean distance": np.mean(liste_dist), 
                            "mean distance to birthplace": np.mean(liste_dist_naiss)}
                df_to_complete = pd.concat([df_to_complete, pd.DataFrame({k:[v] for k,v in dic_line.items()})])
    return df_to_complete

In [ ]:
def create_plot_and_save(colname, only_en_exercice = True, show = True, save = True):
    df = create_indicators_on_column(colname, only_en_exercice)
    Fig_share = px.line(df, x= "year", y = "share not in Paris", color = colname)
    Fig_dist = px.line(df, x= "year", y = "mean distance", color = colname)
    Fig_birth = px.line(df, x= "year", y = "mean distance to birthplace", color = colname)
    Fig_share.update_layout(plot_bgcolor='white')
    Fig_dist.update_layout(plot_bgcolor='white')
    Fig_birth.update_layout(plot_bgcolor='white')
    if show:
        Fig_share.show()
        Fig_dist.show()
        Fig_birth.show()
    if save:
        Fig_share.write_image(path_to_save + 'share travelling.png')
        Fig_dist.write_image(path_to_save + 'Mean distance to Paris.png')
        Fig_birth.write_image(path_to_save + 'Mean distance to birth place.png')
    

In [ ]:
path_to_save

In [ ]:
create_plot_and_save("currently in office", False, save = True)

In [ ]:
create_plot_and_save("francais", True, show= False, save = False)

In [ ]:
create_plot_and_save("senatorerie", True, show= False, save = False)

In [ ]:
create_plot_and_save("previous participation to assembly", True, show= True, save = False)

In [ ]:
create_plot_and_save("other participation to assembly", True, show= True, save = True)

In [ ]:
create_plot_and_save("position", True, show= False, save = False)

In [ ]:
nb_senateurs = pd.DataFrame(columns = ["year", "number of senators"])
for annee in range(1799, 1815):
    senateurs_vivants = senateurs[(senateurs["annee deces"]>annee) & (senateurs["annee nomin"]<=annee)]
    value = len(senateurs_vivants[senateurs_vivants["annee nomin"]<=annee])
    dic_line = {"year" : annee,  "number of senators": value}
    nb_senateurs = pd.concat([nb_senateurs, pd.DataFrame({k:[v] for k,v in dic_line.items()})])

In [ ]:
Fig = px.line(nb_senateurs, x= "year", y = "number of senators")
Fig.update_layout(
    plot_bgcolor='white'
)
Fig.show()
Fig.write_image(path_to_save + 'number senators.png')

In [ ]:
share_foreigners = pd.DataFrame(columns = ["year", "share foreigners"])
for annee in range(1799, 1816):
    senateurs_en_cours = senateurs[(senateurs["annee deces"]>annee) & (senateurs["annee nomin"]<=annee)]
    value = len(senateurs_en_cours[senateurs_en_cours["francais"]==0])/len(senateurs_en_cours)
    dic_line = {"year" : annee,  "share foreigners": value}
    share_foreigners = pd.concat([share_foreigners, pd.DataFrame({k:[v] for k,v in dic_line.items()})])

In [ ]:
Fig = px.line(share_foreigners, x= "year", y = "share foreigners")
Fig.update_layout(plot_bgcolor='white')
Fig.show()
Fig.write_image(path_to_save + 'share foreigners'+'.png')

In [ ]:
share_noble = pd.DataFrame(columns = ["year", "share nobles"])
for annee in range(1799, 1815):
    senateurs_en_cours = senateurs[(senateurs["annee deces"]>annee) & (senateurs["annee nomin"]<=annee)]
    value = len(senateurs_en_cours[senateurs_en_cours["noblesse AR"]=="oui"])/len(senateurs_en_cours)
    dic_line = {"year" : annee,  "share nobles": value}
    share_noble = pd.concat([share_noble, pd.DataFrame({k:[v] for k,v in dic_line.items()})])

In [ ]:
Fig = px.line(share_noble, x= "year", y = "share nobles")
Fig.update_layout(plot_bgcolor='white')
Fig.show()
Fig.write_image(path_to_save + 'share nobles'+'.png')

In [ ]:
True + False

In [ ]:
experience_inter = pd.DataFrame(columns = ["year", "international experience"])
for annee in range(1799, 1816):
    senateurs_en_cours = senateurs[(senateurs["annee deces"]>annee) & (senateurs["annee nomin"]<=annee)]
    internationalise = senateurs_en_cours['international_' + str(annee)] * 0
    for annee2 in list(range(1789, annee))+['naiss', 'AR_1', 'AR_2']:
        internationalise += senateurs['international_' + str(annee2)]
    value = len(senateurs_en_cours[internationalise>0])/len(senateurs_en_cours)
    dic_line = {"year" : annee,  "share internationalized": value}
    experience_inter = pd.concat([experience_inter, pd.DataFrame({k:[v] for k,v in dic_line.items()})])

In [ ]:
Fig = px.line(experience_inter, x= "year", y = "share internationalized")
Fig.update_layout(plot_bgcolor='white', title = 'Share of senators with an international exeprience')
Fig.show()
Fig.write_image(path_to_save + 'share internationalized'+'.png')

In [ ]:
sent_inter = pd.DataFrame(columns = ["year", "sent international"])
for annee in range(1799, 1812):
    senateurs_en_cours = senateurs[(senateurs["annee deces"]>annee) & (senateurs["annee nomin"]==annee)]
    internationalise = senateurs_en_cours['international_' + str(annee)] * 0
    for annee2 in list(range(annee+1, annee+4)):
        internationalise += senateurs['international_' + str(annee2)]
    value = len(senateurs_en_cours[internationalise>0])/len(senateurs_en_cours)
    dic_line = {"year" : annee,  "sent international": value}
    sent_inter = pd.concat([sent_inter, pd.DataFrame({k:[v] for k,v in dic_line.items()})])

In [ ]:
Fig = px.line(sent_inter, x= "year", y = "sent international", 
             labels = {
                 'sent international':  'international in the 3 years'
             })
Fig.update_layout(plot_bgcolor='white', 
                  title = 'Share of new senators with an international destination')
Fig.show()
Fig.write_image(path_to_save + 'sent international'+'.png')

In [ ]:
sent_inter_this_year = pd.DataFrame(columns = ["year", "Not in France"])
for annee in range(1799, 1812):
    senateurs_en_cours = senateurs[(senateurs["annee deces"]>annee) & (senateurs["annee nomin"]==annee)]
    internationalise = senateurs_en_cours['international_' + str(annee)]
    value = len(senateurs_en_cours[internationalise>0])/len(senateurs_en_cours)
    dic_line = {"year" : annee,  "Not in France": value}
    sent_inter_this_year = pd.concat([sent_inter_this_year, pd.DataFrame({k:[v] for k,v in dic_line.items()})])

In [ ]:
Fig = px.line(sent_inter_this_year, x= "year", y = "Not in France", 
             labels = {
                 'Not in France':  'Senator not in France'
             })
Fig.update_layout(plot_bgcolor='white', 
                  title = 'Share of senators that are not in France')
Fig.show()
Fig.write_image(path_to_save + 'not in france'+'.png')

In [ ]:
previous_participation = pd.DataFrame(columns = ["year", "previous participation"])
for annee in range(1799, 1815):
    senateurs_en_cours = senateurs[(senateurs["annee deces"]>annee) & (senateurs["annee nomin"]<=annee)]
    value = np.mean(senateurs_en_cours['previous participation to assembly'])
    dic_line = {"year" : annee,  "previous participation": value}
    previous_participation = pd.concat([previous_participation, pd.DataFrame({k:[v] for k,v in dic_line.items()})])


In [ ]:
Fig = px.line(previous_participation, x= "year", y = "previous participation", 
             labels = {'previous participation':  'participation to an other assembly'})
Fig.update_layout(plot_bgcolor='white', 
                  title = 'Share of senators having participated to an other assembly')
Fig.show()
Fig.write_image(path_to_save + 'previous participation'+'.png')

In [ ]:
position_by_year = pd.DataFrame(columns = ["year"])
for position in senateurs['position'].unique():
    position_by_year[position] = None
for annee in range(1799, 1816):
    dic_line = {"year" : annee}
    for position in senateurs['position'].unique():
        senateurs_en_cours = senateurs[(senateurs["annee deces"]>annee) & (senateurs["annee nomin"]<=annee)]
        value = len(senateurs_en_cours[senateurs_en_cours['position']==position])/len(senateurs_en_cours)
        dic_line[position] = value
    
    position_by_year = pd.concat([position_by_year, pd.DataFrame({k:[v] for k,v in dic_line.items()})])


In [ ]:
path_to_save

In [ ]:
Fig = go.Figure()
for position in senateurs['position'].unique():
    if position == position:
        Fig.add_scatter(x=position_by_year['year'], y=position_by_year[position], mode='lines', name = position)
Fig.update_layout(plot_bgcolor='white', 
                  title = 'Sitting senators background')
Fig.show()
Fig.write_image(path_to_save + 'background senators'+'.png')

In [ ]:
senateurs.columns


In [ ]:
position_by_year = pd.DataFrame(columns = ["year"])
for position in senateurs['activite1800'].unique():
    position_by_year[position] = None
for annee in range(1799, 1816):
    dic_line = {"year" : annee}
    for position in senateurs['activite'+str(annee)].unique():
        senateurs_en_cours = senateurs[(senateurs["annee deces"]>annee) & (senateurs["annee nomin"]<=annee)]
        value = len(senateurs_en_cours[senateurs_en_cours['activite'+str(annee)]==position])/len(senateurs_en_cours)
        dic_line[position] = value
    
    position_by_year = pd.concat([position_by_year, pd.DataFrame({k:[v] for k,v in dic_line.items()})])


In [ ]:
Fig = go.Figure()
for position in senateurs['activite1800'].unique():
    if position == position:
        Fig.add_scatter(x=position_by_year['year'], y=position_by_year[position], mode='lines', name = position)
Fig.update_layout(plot_bgcolor='white', 
                  title = 'Current activity of senators')
Fig.show()
Fig.write_image(path_to_save + 'activity senators'+'.png')

In [ ]:
# Regarder les âges moyens à la nomination, la répartition des types... des nouveaux sénateurs par an

In [ ]:
# régresser la distance sur le fait d'être nommé avec effet fixe individu ?

In [ ]:
# proche d'un endroit où la personne était déjà allé auparavant ? 
#-> récupérer l'endroit précédent le plus proche, rapport de distance >90%

In [ ]:
age_moyen_nouveau = {}
for annee in years:
    age_moyen_nouveau[annee] = annee - np.mean(senateurs[senateurs["annee nomin"]==annee]["annee naiss"])

In [ ]:
Fig = px.line(x = age_moyen_nouveau.keys(), y = age_moyen_nouveau.values(),
             labels={
                     "x": "year",
                     "y": "mean age of new senators"
                 })
Fig.show()
#Fig.write_image(path_to_save + 'mean age new senators'+'.png')

In [ ]:
import bs4
import pandas as pd
from urllib import request
import re
import geopandas as gpd
import networkx as nx
import numpy as np
from mpl_toolkits.basemap import Basemap as Basemap
from pycountry_convert import country_alpha2_to_continent_code, country_name_to_country_alpha2
import geopy
from geopy.geocoders import Nominatim
from matplotlib import pyplot as plt
from ast import literal_eval
import plotly.graph_objs as go
import plotly.express as px
import math



def graph_era(b,e):
    '''
    Tracer le graphique des proportions de mathématiciens par pays entre deux dates
    '''
    #à l'aide de is_alive, on récupère les pays des mathématiciens vivants entre deux dates
    active = df_math_doc.loc[df_math_doc.index.to_series().apply(lambda name : is_alive(b,e,name))==True,
                    ['Country of citizenship']] 
    
    #on crée un dictionnaire qui recense les pays et le nombre de mathématicien qui y a travaillé sur la période
    d_cntry = {}
    for countries in active['Country of citizenship'].values:
        for country in countries:
            count = d_cntry.get(country,0) #on obtient le nombre de mathématiciens pour ce pays ou on crée une nouvelle entrée
            d_cntry[country] = count + 1 #on ajoute le nouveau mathématicien
    df_countries = pd.DataFrame(d_cntry.values(),index=d_cntry.keys(),columns=['Number of mathematicians'])
    df_countries = df_countries.reset_index().rename(columns={'index':'Country'})
    
    #on crée freq pour obtenir des fréquences au lieu des nombres, pour pouvoir faire des comparaisons
    #df_countries['Freq of mathematicians'] = df_countries['Number of mathematicians']/df_countries['Number of mathematicians'].sum()
    df_countries['log # of mathematicians'] = df_countries['Number of mathematicians'].apply(lambda x : math.log(1+x))
    #on veut à présent construire une carte
    #pour cela, on crée un geodataframe
    world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
    world=world[['name','geometry']] #récupération des seules colonnes intéressantes
    world = world.rename(columns={'name':'Country'})
    
    df_countries = world.merge(df_countries,on='Country')
    
    #on trace la carte
    fig = px.choropleth(df_countries,
             geojson = df_countries.geometry,
             locations="Country", 
             locationmode = 'country names',
             color= 'log # of mathematicians',
            #color = 'Freq of mathematicians',
             color_continuous_scale =px.colors.sequential.Sunsetdark,
             #range_color=[0,1],
             hover_name = "Country",
             title='<br>Countries of work of mathematicians from %s to %s'%(b,e))
    return fig,df_countries
    